Title of this file may need to change.
This file will calculate the 3rd and 4th columns that are needed for ARCiS to function

In [ ]:
import numpy as np
from astropy import units as u
import csv
import configparser
import os
import subprocess
from spectres import spectres

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
from matplotlib.pyplot import figure, show, subplots
from matplotlib import pyplot as plt

#import yaml
from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True  # Keep quotes if they exist
yaml.indent(mapping=2, sequence=4, offset=2)  # Preserve indentation
yaml.representer.add_representer(
    float, lambda dumper, data: dumper.represent_scalar("tag:yaml.org,2002:float", str(data))
)
with open("../config/masterconfig.yaml", "r") as file:
    config = yaml.load(file) or {}
integrationtimes = config['integrationtimes']
observingdistances = config['observingdistances']

In [ ]:
def ARCiS_file_generator(ARCiS_filename):
    """Take one LIFEsim spectrum, add colums for noise and bandwidth, and save it in ARCiS folder"""

    # the length of the filename differs between 350 and 10 observing hours, correct for this
    if ARCiS_filename.split('_')[-1] == '350.txt':
        index = 8
    elif ARCiS_filename.split('_')[-1] == '10.txt':
        index = 7
    
    # get clean LIFEsim data [Jy]
    data_path = "../output/LIFEsim/clean_spectra/textfiles/" + ARCiS_filename[9:-index]
    bandcenter = np.genfromtxt(data_path,skip_header=0)[:,0] # microns
    bandflux = np.genfromtxt(data_path,skip_header=0)[:,1] # Jansky

    # get observed LIFEsim noise and bandwidths
    data_path = "../output/LIFEsim/observed_spectra/textfiles/" + ARCiS_filename
    bins = np.genfromtxt(data_path,skip_header=2,delimiter=",")[:,0] # microns
    SN = np.genfromtxt(data_path,skip_header=2,delimiter=",")[:,1] # Jansky
    
    # calculate spectral resolution per bin; should be very close to LIFEsim R
    spectral_resolution = []
    for i in range(len(bins)):
        if i == 0:
            spectral_resolution.append(bins[i]/(bins[i+1]-bins[i]))
        else:
            spectral_resolution.append(bins[i]/(bins[i]-bins[i-1]))

    # change original AGNI spectrum to LIFEsim bins
    new_bandflux = spectres(bins,bandcenter,bandflux)

    # draw random sample using LIFEsim SN
    measurements = np.random.normal(new_bandflux,new_bandflux/SN,size=new_bandflux.shape)

    fig = figure(figsize=(8,6))
    frame1 = fig.add_subplot(1,1,1)
    frame1.plot(bins,new_bandflux,color='black', label='Input Spectrum')
    frame1.set_xlabel('wavelength (microns)')
    frame1.set_ylabel('Janskys')
    frame1.set_yscale('log')
    #frame1.set_ylim(plotmin,plotmax)
    frame1.fill_between(bins,
                 new_bandflux-new_bandflux/SN,
                 new_bandflux+new_bandflux/SN,
                 color='black', alpha=0.1, label='1-$\sigma$')
    skip_values = 5
    #frame1.scatter(bins[0::skip_values], new_bandflux[0::skip_values]+noise[0::skip_values],
    #        color='black', marker='.', label='Sim. Observation')
    frame1.errorbar(bins[0::skip_values], measurements[0::skip_values],
             yerr=abs(new_bandflux[0::skip_values]/SN[0::skip_values]), color='black', capsize=4, fmt='o',label='Simulated measurements')
    frame1.legend()

    # save images
    plt.ioff() # supress notebook output
    filename = "../output/ARCiS/input_spectra/figures/" + "ARCiS_fm_input_" + ARCiS_filename[17:-4] + '.png'
    fig.savefig(filename)
    
    
    # write to text files
    file_destination = "../output/ARCiS/input_spectra/textfiles/" + "ARCiS_fm_input_" + ARCiS_filename[17:-4] + ".txt"
    with open(file_destination, 'w') as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerows(zip(bins,measurements,new_bandflux/SN,spectral_resolution))
    

In [ ]:
data_path = "../output/LIFEsim/observed_spectra/textfiles"
files = [f for f in os.listdir(data_path) if os.path.isfile(os.path.join(data_path, f))]
for i in range(len(files)):
    ARCiS_file_generator(files[i])

In [ ]:
files = os.listdir("../output/ARCiS/input_spectra/textfiles/")

# sort files first by fo2, then CH then distance
def sort_key(name):
    parts = name.replace("ARCiS_fm_input_", "").split("_")
    fo2 = float(parts[0])
    CH = float(parts[1])
    dist = float(parts[2])
    return (fo2, CH, dist)
files_sorted = sorted(files, key=sort_key)

mpl.rcParams['figure.dpi'] = 800
custom_order = [9,5,1,10,6,2,11,7,3,12,8,4]

def plot_LIFEsim_noise(select_distance, select_observingtime):
    rasterplot = figure(figsize=(8,6))
    custom_count = 0
    for i in files_sorted:
        if i.split("_")[-2] == str(select_distance):
            if i.split("_")[-1] == str(select_observingtime) + '.txt':
                frame = rasterplot.add_subplot(3,4,custom_order[custom_count])

                # get clean LIFEsim data [Jy]
                if select_observingtime == 350:
                    data_path = "../output/LIFEsim/clean_spectra/textfiles/fm_ARCiS" + i[15:-8]
                else:
                    data_path = "../output/LIFEsim/clean_spectra/textfiles/fm_ARCiS" + i[15:-7]
                bandcenter = np.genfromtxt(data_path,skip_header=0)[:,0] #[745:1045] # microns
                bandflux = np.genfromtxt(data_path,skip_header=0)[:,1]# [745:1045] # Jansky
                

                path = "../output/ARCiS/input_spectra/textfiles/" + i
                band = np.genfromtxt(path,usecols=0)
                emis = np.genfromtxt(path,usecols=1)
                noise = np.genfromtxt(path,usecols=2)

                new_bandflux = spectres(band,bandcenter,bandflux)

                frame.plot(band,new_bandflux,color='blue', label='Input Spectrum',linewidth=0.5)
                frame.fill_between(band,
                            new_bandflux-noise,
                            new_bandflux+noise,
                            color='black', alpha=0.3, label='1-$\sigma$',edgecolor='none')
                
                #frame.set_xlabel('wavelength (microns)')
                #frame.set_ylabel('Janskys')
                frame.set_yscale('log')
                if select_distance == 5:
                    frame.set_ylim(5e-10,5e-06)
                elif select_distance == 10:
                    frame.set_ylim(10e-11,1.8e-06)
                elif select_distance == 15:
                    frame.set_ylim(5e-11,8e-7)
                elif select_distance == 20:
                    frame.set_ylim(2e-11,5e-07)

                #frame.set_ylim(1.25e-06)
                #frame.set_yscale('symlog', linthresh=1e-7)
                #frame1.set_ylim(plotmin,plotmax)
                #frame1.fill_between(bins,
                                # new_bandflux-new_bandflux/SN,
                                # new_bandflux+new_bandflux/SN,
                                # color='black', alpha=0.1, label='1-$\sigma$')
                skip_values = 5
                #frame1.scatter(bins[0::skip_values], new_bandflux[0::skip_values]+noise[0::skip_values],
                #        color='black', marker='.', label='Sim. Observation')
                frame.errorbar(band[0::skip_values], emis[0::skip_values],
                                yerr=abs(noise[0::skip_values]), color='black', fmt='o',label='Simulated measurements',
                                capsize=2, markersize=1.5,elinewidth=0.7,capthick=0.7)
                plt.tick_params(
                    axis='both',      # both x and y axes
                    which='both',     # major and minor ticks
                    labelsize=5.5,      # ↓ font size of tick labels
                    length=2,         # ↓ length of tick marks
                    width=0.4,         # ↓ thickness of tick marks
                    pad=1.2
                )
                frame.minorticks_off()
                frame.set_xlim(4,18.5)
                custom_count += 1


    # add axes to the plot (I feel like I over-engineered this)
    master_ax = rasterplot.add_axes([0.09, 0.08, 0.82, 0.82])
    master_ax.patch.set_alpha(0)
    master_ax.set_zorder(-10)

    master_ax.spines['top'].set_visible(False)
    master_ax.spines['right'].set_visible(False)
    master_ax.spines['left'].set_linewidth(1.5)
    master_ax.spines['bottom'].set_linewidth(1.5)

    y_centers = [0.83, 0.50, 0.17] 
    master_ax.set_ylim(0, 1)
    master_ax.set_yticks(y_centers)
    master_ax.set_yticklabels(["5", "1", "0.1"])

    x_centers = [0.135, 0.365, 0.635, 0.865]
    master_ax.set_xlim(0, 1)
    master_ax.set_xticks(x_centers)
    master_ax.set_xticklabels(["-4", "-1.3", "1.3", "4"])

    master_ax.tick_params(labelsize=12)


    rasterplot.supxlabel('Mantle $f$O$_2$, $\Delta$ IW')
    rasterplot.supylabel('Mantle C/H mass ratio')
    #rasterplot.suptitle("AGNI spectra for varying mantle $f$O$_2$ and mantle H/C mass ratio")
    rasterplot.show()
    rasterplot.savefig('../output/figures/LIFEsim_observednoise_raster_{}pc_{}hours.png'.format(select_distance,select_observingtime))

In [ ]:
for select_distance in observingdistances:
    for select_observingtime in [round(i/60/60) for i in integrationtimes]:
        print(select_distance,select_observingtime)
        plot_LIFEsim_noise(select_distance, select_observingtime)
